# 04 – NLP Analysis

### Purpose of the Notebook
Analyse textbasierter Vergabefelder mittels NLP.

### Steps
- TF‑IDF preprocessing
- SVD dimensionality reduction
- NMF topic modelling
- SVM text‑risk classifier (3 classes)
- Export TEXT_RISK_SCORE + NLP features
- Integration into modelling pipeline

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

import pickle

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.nlp_processing import (text_preprocessing, build_tfidf_svd, build_nmf_topics,
                               build_text_risk_classifier, predict_text_risk, map_risk_from_offers)

from my_scripts.eda import (overview, filter_germany)

In [5]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset_fe.pkl")
df = df.reset_index(drop=True)

print("EU dataset:", df.shape)

EU dataset: (3521623, 32)


----------------
### NLP PROCESSING

----------

In [7]:
# ---------------------------------------------------------
# Preprocess text 
# ---------------------------------------------------------

df = text_preprocessing(df, ["TEXT_ALL"])


In [8]:
df.shape

(3521623, 32)

In [9]:
# ---------------------------------------------------------
# TF‑IDF + SVD features
# ---------------------------------------------------------

df_svd, X_tfidf, tfidf_vectorizer, svd_model = build_tfidf_svd(df["TEXT_ALL"])
df = pd.concat([df, df_svd], axis=1)


In [10]:
# ---------------------------------------------------------
# Topic modelling (NMF)
# ---------------------------------------------------------
df_topics, nmf_model = build_nmf_topics(X_tfidf)
df = pd.concat([df, df_topics], axis=1)


In [11]:
df.shape

(3521623, 147)

In [12]:
# ---------------------------------------------------------
# Labels for TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_LABEL"] = df["NUMBER_OFFERS"].apply(map_risk_from_offers)


In [13]:
# ---------------------------------------------------------
# Train SVM classifier
# ---------------------------------------------------------

df_text = df[df["TEXT_RISK_LABEL"].notna()].copy()

texts = df_text["TEXT_ALL"].fillna("").astype(str)
labels = df_text["TEXT_RISK_LABEL"].astype(str)

svm_model, tfidf_svm, label_encoder = build_text_risk_classifier(texts, labels)



In [14]:
# ---------------------------------------------------------
# Predict TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_SCORE"] = predict_text_risk(
    df["TEXT_ALL"],
    svm_model,
    tfidf_svm,
    label_encoder
)



In [15]:
df.head()

,YEAR,ID_TYPE,CANCELLED,CORRECTIONS,ISO_COUNTRY_CODE,CAE_TYPE,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,B_DYN_PURCH_SYST,LOTS_NUMBER,VALUE_EURO,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,B_AWARDED_TO_A_GROUP,B_CONTRACTOR_SME,NUMBER_OFFERS,AWARD_VALUE_EURO,B_SUBCONTRACTED,TEXT_ALL,IS_FAILED_TENDER,AWARD_MONTH,AWARD_QUARTER,DAYS_TO_AWARD,VALUE_EURO_MISSING,AWARD_VALUE_EURO_MISSING,CPV_CATEGORY,NLP_SVD_0,NLP_SVD_1,NLP_SVD_2,NLP_SVD_3,NLP_SVD_4,NLP_SVD_5,NLP_SVD_6,NLP_SVD_7,NLP_SVD_8,NLP_SVD_9,NLP_SVD_10,NLP_SVD_11,NLP_SVD_12,NLP_SVD_13,NLP_SVD_14,NLP_SVD_15,NLP_SVD_16,NLP_SVD_17,NLP_SVD_18,NLP_SVD_19,NLP_SVD_20,NLP_SVD_21,NLP_SVD_22,NLP_SVD_23,NLP_SVD_24,NLP_SVD_25,NLP_SVD_26,NLP_SVD_27,NLP_SVD_28,NLP_SVD_29,NLP_SVD_30,NLP_SVD_31,NLP_SVD_32,NLP_SVD_33,NLP_SVD_34,NLP_SVD_35,NLP_SVD_36,NLP_SVD_37,NLP_SVD_38,NLP_SVD_39,NLP_SVD_40,NLP_SVD_41,NLP_SVD_42,NLP_SVD_43,NLP_SVD_44,NLP_SVD_45,NLP_SVD_46,NLP_SVD_47,NLP_SVD_48,NLP_SVD_49,NLP_SVD_50,NLP_SVD_51,NLP_SVD_52,NLP_SVD_53,NLP_SVD_54,NLP_SVD_55,NLP_SVD_56,NLP_SVD_57,NLP_SVD_58,NLP_SVD_59,NLP_SVD_60,NLP_SVD_61,NLP_SVD_62,NLP_SVD_63,NLP_SVD_64,NLP_SVD_65,NLP_SVD_66,NLP_SVD_67,NLP_SVD_68,NLP_SVD_69,NLP_SVD_70,NLP_SVD_71,NLP_SVD_72,NLP_SVD_73,NLP_SVD_74,NLP_SVD_75,NLP_SVD_76,NLP_SVD_77,NLP_SVD_78,NLP_SVD_79,NLP_SVD_80,NLP_SVD_81,NLP_SVD_82,NLP_SVD_83,NLP_SVD_84,NLP_SVD_85,NLP_SVD_86,NLP_SVD_87,NLP_SVD_88,NLP_SVD_89,NLP_SVD_90,NLP_SVD_91,NLP_SVD_92,NLP_SVD_93,NLP_SVD_94,NLP_SVD_95,NLP_SVD_96,NLP_SVD_97,NLP_SVD_98,NLP_SVD_99,NLP_TOPIC_0,NLP_TOPIC_1,NLP_TOPIC_2,NLP_TOPIC_3,NLP_TOPIC_4,NLP_TOPIC_5,NLP_TOPIC_6,NLP_TOPIC_7,NLP_TOPIC_8,NLP_TOPIC_9,NLP_TOPIC_10,NLP_TOPIC_11,NLP_TOPIC_12,NLP_TOPIC_13,NLP_TOPIC_14,TEXT_RISK_LABEL,TEXT_RISK_SCORE
0,2008,3,0,0,DE,8,Unknown,W,0,0.00,"512,637.02",Unknown,OPE,0,0,M,100.00,0,1,0,0,2.00,"302,964.15",Unknown,preis qualit t 60 40,0,9.00,3.00,-73.00,1,0,Other,0.23,-0.01,-0.30,0.02,-0.03,0.00,0.29,0.15,-0.06,-0.04,-0.23,-0.04,0.09,0.15,-0.01,0.05,-0.07,-0.05,-0.06,0.07,0.04,0.13,0.12,-0.20,0.10,-0.07,-0.04,-0.00,-0.05,-0.06,0.05,-0.08,-0.01,0.02,-0.02,0.03,0.05,-0.00,-0.01,-0.03,0.02,0.01,0.02,-0.03,0.00,-0.03,-0.02,0.07,0.01,-0.01,0.02,-0.02,-0.07,0.05,0.03,0.04,0.06,0.06,0.09,0.01,0.01,-0.20,0.06,-0.11,-0.01,-0.01,0.09,0.01,-0.02,-0.02,-0.04,-0.03,-0.06,0.10,0.02,-0.07,-0.03,-0.13,0.00,0.05,0.04,-0.05,0.04,-0.01,-0.01,-0.11,0.03,-0.01,-0.01,-0.04,0.03,-0.05,0.05,0.05,0.00,-0.07,0.02,0.08,-0.01,0.08,0.00,0.00,0.04,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.00,medium,low
1,2008,3,0,0,DE,3,Unknown,W,0,0.00,"512,637.02",N,OPE,0,0,L,100.00,0,1,0,0,3.00,"478,780.08",N,,0,12.00,4.00,-6.00,1,0,Other,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,medium,medium
2,2008,3,0,0,FR,3,Unknown,W,0,0.00,"512,637.02",N,OPE,0,0,M,100.00,0,1,0,0,1.00,"36,610.02",N,prix d lai d intervention d urgence 80 20,1,11.00,4.00,-37.00,1,1,Other,0.10,0.00,0.02,0.07,-0.02,-0.00,-0.00,-0.03,0.01,0.19,0.09,0.04,0.31,-0.07,-0.02,-0.03,-0.05,-0.04,-0.03,0.04,-0.03,-0.04,-0.02,0.02,0.01,-0.04,-0.05,-0.00,0.01,-0.01,-0.01,-0.02,-0.01,-0.00,-0.02,-0.00,-0.01,-0.06,0.02,0.03,0.06,0.00,0.07,0.00,0.10,0.05,0.03,0.01,0.08,-0.00,-0.07,0.02,-0.04,-0.01,-0.01,0.05,0.05,-0.00,0.02,0.00,-0.02,-0.05,-0.08,0.09,-0.01,0.01,-0.08,0.01,-0.00,-0.01,0.02,0.02,0.02,0.02,0.03,-0.03,0.01,-0.03,0.00,0.01,0.01,0.03,0.01,0.03,-0.03,0.02,-0.01,-0.04,0.02,-0.03,-0.03,0.00,0.01,0.01,0.01,0.01,0.00,-0.02,-0.02,0.00,0.00

In [16]:
df.shape

(3521623, 149)

In [17]:
df.columns

Index(['YEAR', 'ID_TYPE', 'CANCELLED', 'CORRECTIONS', 'ISO_COUNTRY_CODE',
       'CAE_TYPE', 'B_AWARDED_BY_CENTRAL_BODY', 'TYPE_OF_CONTRACT',
       'B_DYN_PURCH_SYST', 'LOTS_NUMBER',
       ...
       'NLP_TOPIC_7', 'NLP_TOPIC_8', 'NLP_TOPIC_9', 'NLP_TOPIC_10',
       'NLP_TOPIC_11', 'NLP_TOPIC_12', 'NLP_TOPIC_13', 'NLP_TOPIC_14',
       'TEXT_RISK_LABEL', 'TEXT_RISK_SCORE'],
      dtype='str', length=149)

In [19]:
# drop unnecessary column
df = df.drop(columns="TEXT_ALL", errors="ignore").reset_index(drop=True)


#### Notes: NLP Pipeline for Tender Risk Prediction

1. Dataset Size After Features Engineering
- Full EU dataset:
  - Before: 3,521,623 rows × 32 columns
  - After: 3,521,623 rows × 149 columns

2. Text fields provide additional signals not captured by structured variables.  
Short titles and evaluation‑related text (TITLE, CRIT_CRITERIA, CRIT_WEIGHTS AS TEXT_ALL) reveal complexity, niche requirements, and multi‑criteria scoring patterns that strongly influence bidder participation and failure risk.

3. TF‑IDF offers a scalable and domain‑appropriate representation of tender text.  
It efficiently captures important terms and patterns without requiring heavy linguistic models, making it suitable for millions of records and classical ML workflows.

4. Dimensionality reduction (SVD) converts high‑dimensional TF‑IDF vectors into compact numerical features.  
This reduces sparsity, stabilizes downstream models, and enables seamless integration with structured predictors such as CPV, procedure type, and tender value.

5. Topic modelling (NMF) introduces interpretable thematic structure.  
Extracted topics highlight procurement areas with systematically higher failure rates, improving interpretability and analytical insight.

6. A text‑based classifier (TF‑IDF + SVM) produces a high‑level TEXT_RISK_SCORE.  
It learns patterns associated with failed, medium‑risk, and safe tenders based solely on text, generating a categorical risk signal usable even when raw text is unavailable.

7. TEXT_RISK_SCORE is essential for downstream applications such as the risk simulator.  
It allows the model to incorporate text‑derived risk information without requiring free‑text input, enabling scenario simulations based on a small set of structured parameters.

8. The combined pipeline remains interpretable, scalable, and robust.  
TF‑IDF + SVD ensures numerical stability, NMF adds thematic insight, and SVM provides a practical risk score — together forming a balanced NLP module that strengthens the overall tender risk prediction model.

--------------
## Top topics

-------------

In [20]:
# ---------------------------------------------------------
# General Topics
# ---------------------------------------------------------

#choice topics

feature_names = tfidf_vectorizer.get_feature_names_out()

topics = {}

for i, topic in enumerate(nmf_model.components_):
    top_indices = topic.argsort()[-20:]  # топ 20 слів
    top_words = [feature_names[j] for j in top_indices]
    topics[f"Topic_{i}"] = top_words


In [21]:
# print topics
for t, words in topics.items():
    print(f"{t}: {', '.join(words)}")


Topic_0: crit, technique de, livraison, qualit, lot, au, de offre, re, sur, offre, les, pour, le, en, de la, du, des, la, et, de
Topic_1: nr 20, 16, nr 16, pozycja, 14, 13, nr 14, 12, nr 13, nr 15, 11, nr poz, nr 12, nr 11, nr cena, nr 10, poz, nr, pakiet nr, pakiet
Topic_2: qualit 60, prezzo 60, pris, prestations 60, 40 40, jako 60, prix valeur, valeur, valeur technique, technique, quality 60, prix, price 60, technique prix, technique 60, 40 60, prix 60, 60 40, 40, 60
Topic_3: services, to, technical, lot, 40 60, service, quality 70, price 70, cost, delivery, quality 40, the, quality 60, of, price 60, and, quality price, price quality, price, quality
Topic_4: cz, termin realizacji, do, realizacji, 99, ci 95, dostawy 95, 90 10, 90, 98, ci, termin atno, atno ci, atno, 95, termin dostawy, dostawy, cena termin, termin, cena
Topic_5: og lne, leki og, ne cena, leki psychotropowe, onkologiczne, leki onkologiczne, leki stosowane, psychotropowe, stosowane, zad, pakiet, ce, pakiet leki, nr leki

#### Notes: Interpreted Topic Categories
1. Topic 0 — Tender Lots & Procurement Packages (PL/NL)
  - Frequent terms: pozycja, grupa, pakiet, zadanie, dostawa, poz, perceel  
  - Meaning: structural elements of tenders (lots, packages, tasks).
  - Category: Tender structure / lot definitions

2. Topic 1 — Technical Supply & Delivery Conditions (FR)
  - Frequent terms: fourniture, livraison, prix de, offre prix, technique de  
  - Meaning: French technical descriptions and delivery specifications.
  - Category: Technical supply & delivery requirements

3. Topic 2 — Lot Numbering & Sub‑lot Structure (PL/NL)
  - Frequent terms: nr, perceel nr, zadanie nr, pakiet nr  
  - Meaning: numbering of lots, sub‑lots, and tender sections.
  - Category: Lot numbering / sub‑lot segmentation

4. Topic 3 — Price vs Technical Value (FR)
  - Frequent terms: offre prix, technique, valeur technique, prix 60  
  - Meaning: French scoring formulas combining price and technical value.
  - Category: Price–technical value scoring

5. Topic 4 — Quality–Price Evaluation (EN)
  - Frequent terms: technical, quality, delivery, price 70, quality 40  
  - Meaning: English‑language tenders evaluating quality and price.
  - Category: Quality–price evaluation (EN tenders)

6. Topic 5 — Delivery Deadlines & Execution Timing (PL)
  - Frequent terms: termin realizacji, dostawy, cena termin, 90/10  
  - Meaning: deadlines, delivery schedules, execution timing.
  - Category: Delivery timelines / execution deadlines

7. Topic 6 — German Technical Criteria (Preis/Qualität)
  - Frequent terms: preis, technischer wert, qualit, los, preis qualit  
  - Meaning: German scoring based on technical value and price.
  - Category: German technical criteria (Preis/Qualität)

8. Topic 7 — Scoring Formulas (Mixed Languages)
  - Frequent terms: 30/10, 40/30, cena, jakość, price 70  
  - Meaning: mixed scoring formulas (30/70, 40/60, 70/30).
  - Category: Scoring ratios (price/quality)

9. Topic 8 — Price–Quality Scoring (IT/FR)
  - Frequent terms: prezzo, qualit, prix 50, 50/50  
  - Meaning: Italian/French scoring formulas.
  - Category: Price–quality scoring (IT/FR)

10. Topic 9 — Pharmaceuticals & Medical Drugs (PL)
  - Frequent terms: leki onkologiczne, psychotropowe, pakiet leki  
  - Meaning: medical and pharmaceutical procurement.
  - Category: Pharmaceutical products / medical drugs

11. Topic 10 — Service & Management Contracts (EN)
  - Frequent terms: delivery, management, service, 40/10, 30/10  
  - Meaning: service‑based tenders, management contracts.
  - Category: Service & management procurement

12. Topic 11 — Quality–Price Evaluation (FR/IT)
  - Frequent terms: prix, qualit, prezzo, technique 60, 40/60  
  - Meaning: quality/price scoring in FR/IT tenders.
  - Category: Quality–price evaluation (FR/IT)

13. Topic 12 — Technical Services & Performance Criteria (FR)
  - Frequent terms: prestations, technique des, valeur technique  
  - Meaning: technical services, performance‑based evaluation.
  - Category: Technical services / performance criteria

14. Topic 13 — High‑Weight Quality Scoring (80/20 etc.)
  - Frequent terms: cena jakość, quality 80, technique 80, 40/40  
  - Meaning: tenders with strong emphasis on quality.
  - Category: High‑weight quality scoring

15. Topic 14 — General Evaluation Criteria (FR)
  - Frequent terms: critère, prix, qualité, lots, de la, pour les  
  - Meaning: general French evaluation criteria.
  - Category: General evaluation criteria (FR)


In [22]:
# ---------------------------------------------------------
# Top Topics for Germany
# ---------------------------------------------------------

# Germany dataset
df_de = filter_germany(df)

# topics range for Germany

topic_cols = [c for c in df_de.columns if c.startswith("NLP_TOPIC_")]
df_de["dominant_topic"] = df_de[topic_cols].idxmax(axis=1)
df_de["dominant_topic"].value_counts()

dominant_topic
NLP_TOPIC_0     99109
NLP_TOPIC_6     69515
NLP_TOPIC_9     28354
NLP_TOPIC_10    15379
NLP_TOPIC_12    12388
NLP_TOPIC_8     11225
NLP_TOPIC_7      8728
NLP_TOPIC_2      6581
NLP_TOPIC_14     1951
NLP_TOPIC_11     1778
NLP_TOPIC_13     1190
NLP_TOPIC_3       841
NLP_TOPIC_4       339
NLP_TOPIC_5       158
NLP_TOPIC_1        38
Name: count, dtype: int64

### Germany‑Specific Topic Profile (based on PCA, KMeans, and dominant topic distribution)
1. Tender Lots & Procurement Packages (Topic 0)
2. Technical Criteria (Topic 6)
3. Pharmaceuticals & Medical Drugs (Topic 9)
4. Service & Management Contracts (Topic 10)
5. Technical Services & Performance Criteria (Topic 12)
6. Price–Quality Scoring (IT/FR patterns) (Topic 8)
7. Scoring Formulas (price/quality ratios) (Topic 7)
8. Lot Numbering & Sub‑lot Structure (Topic 2)
9. General Evaluation Criteria (Topic 14)
10. Quality–Price Evaluation (FR/IT) (Topic 11)

--------------
### SAVE DATASET & MODEL

--------------

In [23]:
# Topics for Germany 

df_de = filter_germany(df)

topic_cols = [c for c in df_de.columns if c.startswith("NLP_TOPIC_")]

df_de_topics = df_de[topic_cols].copy()

df_de_topics.to_pickle("../data/dataset_topics_de.pkl")

In [24]:
# save dataset
df.to_pickle("../data/dataset_nlp.pkl")

In [25]:
# save modell

with open("../models/nmf_model.pkl", "wb") as f:
    pickle.dump(nmf_model, f)

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)
